# MCQA Filtering Diagnostic

This notebook inspects the exact next-token filtering path used by the local MCQA code.
It loads the public HF dataset, shows the predicted next token for sampled prompts, and checks whether each example would be kept by the filter.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
REPO_ROOT = cwd.parent if cwd.name == "mcqa_experiment" else cwd
sys.path.insert(0, str(REPO_ROOT))

MODEL_NAME = "google/gemma-2-2b"
DATASET_PATH = "jchang153/copycolors_mcqa"
DATASET_CONFIG = None
RAW_SIZE_CAP = 100
BATCH_SIZE = 16
DATASET_KEY = None
EXAMPLE_INDEX = 0
DEVICE = None


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from mcqa_experiment.data import (
    MCQACausalModel,
    _encode_symbol_token_variants,
    _infer_next_token_ids,
    filter_correct_examples,
    load_public_mcqa_datasets,
    normalize_answer_text,
)
from mcqa_experiment.runtime import resolve_device

torch_device = resolve_device(DEVICE)
dtype = torch.float16 if torch_device.type in {"cuda", "mps"} else torch.float32

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype,
    attn_implementation="eager",
)
model.to(torch_device)
model.eval()

causal_model = MCQACausalModel()
datasets_by_name = load_public_mcqa_datasets(
    size=RAW_SIZE_CAP,
    dataset_path=DATASET_PATH,
    dataset_name=DATASET_CONFIG,
)

sorted(datasets_by_name.keys())[:10]

In [ ]:
if DATASET_KEY is None:
    DATASET_KEY = sorted(datasets_by_name.keys())[0]

rows = datasets_by_name[DATASET_KEY]
print("dataset:", DATASET_KEY)
print("rows:", len(rows))
rows[0].keys()

In [ ]:
def next_token_debug(prompt: str):
    encoded = tokenizer(prompt, return_tensors="pt", add_special_tokens=True)
    input_ids = encoded["input_ids"].to(torch_device)
    attention_mask = encoded["attention_mask"].to(torch_device)
    predicted_id = int(_infer_next_token_ids(model, input_ids, attention_mask)[0].item())
    decoded = tokenizer.decode([predicted_id])
    generated = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=1,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    continuation_ids = generated[0, input_ids.shape[1]:].detach().cpu().tolist()
    continuation_text = tokenizer.decode(continuation_ids)
    return {
        "predicted_id": predicted_id,
        "decoded_next_token": decoded,
        "generated_ids": continuation_ids,
        "generated_text": continuation_text,
    }


def inspect_example(row):
    base_input = row["input"]
    source_input = row["counterfactual_inputs"][0]
    base_output = causal_model.run_forward(base_input)
    source_output = causal_model.run_forward(source_input)

    expected = normalize_answer_text(str(base_output["answer"]))
    expected_variants = _encode_symbol_token_variants(expected, tokenizer)
    debug = next_token_debug(str(base_input["raw_input"]))
    keep = int(debug["predicted_id"]) in expected_variants or normalize_answer_text(debug["decoded_next_token"]) == expected

    print("BASE PROMPT")
    print(base_input["raw_input"])
    print()
    print("SOURCE PROMPT")
    print(source_input["raw_input"])
    print()
    print("EXPECTED BASE ANSWER:", repr(base_output["answer"]), "normalized=", repr(expected))
    print("EXPECTED SOURCE ANSWER:", repr(source_output["answer"]))
    print("EXPECTED VARIANT TOKEN IDS:", expected_variants)
    print("PREDICTED NEXT TOKEN ID:", debug["predicted_id"])
    print("DECODED NEXT TOKEN:", repr(debug["decoded_next_token"]))
    print("GREEDY GENERATE TEXT:", repr(debug["generated_text"]))
    print("FILTER KEEP:", keep)
    return {
        "base_input": base_input,
        "source_input": source_input,
        "base_output": base_output,
        "source_output": source_output,
        "debug": debug,
        "keep": keep,
    }


In [ ]:
inspection = inspect_example(rows[EXAMPLE_INDEX])

In [ ]:
subset = {DATASET_KEY: rows}
filtered_subset = filter_correct_examples(
    model=model,
    tokenizer=tokenizer,
    causal_model=causal_model,
    datasets_by_name=subset,
    batch_size=BATCH_SIZE,
    device=torch_device,
)

kept_rows = filtered_subset[DATASET_KEY]
print("kept", len(kept_rows), "out of", len(rows))

In [ ]:
results = []
for idx, row in enumerate(rows[: min(10, len(rows))]):
    base_output = causal_model.run_forward(row["input"])
    expected = normalize_answer_text(str(base_output["answer"]))
    variants = _encode_symbol_token_variants(expected, tokenizer)
    debug = next_token_debug(str(row["input"]["raw_input"]))
    keep = int(debug["predicted_id"]) in variants or normalize_answer_text(debug["decoded_next_token"]) == expected
    results.append({
        "idx": idx,
        "expected": expected,
        "predicted_id": debug["predicted_id"],
        "decoded": normalize_answer_text(debug["decoded_next_token"]),
        "variant_ids": variants,
        "keep": keep,
    })

results

## Notes

- `DATASET_PATH` defaults here to your public HF dataset: `jchang153/copycolors_mcqa`.
- `RAW_SIZE_CAP` only limits how many raw rows are loaded for diagnosis.
- `inspect_example(...)` is the simplest way to check whether one prompt is being filtered correctly.
- The filter accepts either single-token encoding for the expected answer symbol, so `"A"` and `" A"` are treated as equivalent when tokenized to one token.